## 1. Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import r2_score

pd.set_option("display.max_columns", 1000)

## 2. Data Loading & Cleaning

In [ ]:
df = pd.read_csv('data/clean/final_features.csv')

df = df.drop(columns=[
    "state",
    "date",
    "state_staffing_shortage_anticipation_completeness",
    "state_staffing_reporting_completeness",
    "inpatient_beds_used_covid",
])
df = df.dropna()
df

## 3. Random Forest — Predicting `inpatient_beds_used`

In [ ]:
SEED = 42

X = df.drop(columns='inpatient_beds_used')
y = df['inpatient_beds_used']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=SEED)

rfr = RandomForestRegressor(random_state=SEED)
rfr.fit(X_train, y_train)

### 3.1 Train vs Test R² (Overfitting Check)

In [ ]:
r2_train = r2_score(y_train, rfr.predict(X_train))
r2_test  = r2_score(y_test,  rfr.predict(X_test))

print(f"Train R²: {r2_train:.4f}")
print(f"Test  R²: {r2_test:.4f}")
print(f"Gap:      {r2_train - r2_test:.4f}  (>0.05 suggests overfitting)")

### 3.2 Cross-Validated R²

In [ ]:
cv_scores = cross_val_score(rfr, X, y, cv=5, scoring='r2')

print(f"CV R² scores: {cv_scores.round(4)}")
print(f"Mean: {cv_scores.mean():.4f}  |  Std: {cv_scores.std():.4f}")

### 3.3 Feature Importances

In [ ]:
importances = pd.Series(rfr.feature_importances_, index=X.columns)
importances = importances.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, len(importances) * 0.35))
importances.plot(kind='barh', ax=ax)
ax.set_title('Random Forest Feature Importances')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()